# 🎬 Manim Video Tool — Course Planner

Render a short **math / physics** video with **Manim Community**, get a clean **MP4**, then commit it to your site's `videos/` folder so it shows up in the **Video Library**.

## Before you start — add your Gemini key as a Colab *secret*
1. Click the **🔑 key icon** in the left sidebar ("Secrets").
2. Click **"+ Add new secret"**.
3. Name it exactly **`GEMINI_API_KEY`** and paste your free key from [Google AI Studio](https://aistudio.google.com/app/apikey).
4. Toggle **"Notebook access" ON** for it.

The key is read with `userdata.get('GEMINI_API_KEY')` and is **never written into the notebook or its output**.

## How to use
Run the cells top to bottom:
1. **Install** Manim + LaTeX (takes a few minutes).
2. **Two smoke tests** confirm Manim renders *and* that LaTeX equations render — these are different bars.
3. Edit the **inputs** cell (prompt, subject, template).
4. Run the **render** cell — it fills a vetted scene template with Gemini, renders, and auto-repairs render errors up to 3 times.
5. Run the **save** cell to send the MP4 to Drive.

## 1. Install Manim + LaTeX
Pinned on purpose. Bump `MANIM_VERSION` deliberately, then re-run the smoke tests.

In [ ]:
# Pinned versions — bump deliberately, then re-run the smoke tests below.
MANIM_VERSION = "0.18.1"

print("Installing system LaTeX + Cairo/Pango/ffmpeg (a few minutes)...")
!apt-get -qq update > /dev/null
!apt-get -qq install -y libcairo2-dev libpango1.0-dev ffmpeg dvisvgm texlive texlive-latex-extra texlive-fonts-extra texlive-latex-recommended texlive-science tipa > /dev/null
print("Installing Manim Community", MANIM_VERSION, "...")
!pip -q install "manim==$MANIM_VERSION"
print("\nDone. Manim version:")
!manim --version


## 2. Connect Gemini
Reads your Colab secret and does a 1-line reachability check.

In [ ]:
import os, re, json, glob, subprocess
import requests

try:
    from google.colab import userdata
    API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    API_KEY = os.environ.get("GEMINI_API_KEY")

assert API_KEY, "No Gemini key found. Add a Colab secret named GEMINI_API_KEY (see the top cell)."

# Free-tier models. 2.5-flash is the reliable default; gemini-3-flash-preview is often
# stronger at code but is a preview id that could be renamed.
MODEL = "gemini-2.5-flash"

def gemini(prompt, as_json=False, temperature=0.4):
    url = "https://generativelanguage.googleapis.com/v1beta/models/{}:generateContent".format(MODEL)
    body = {"contents": [{"parts": [{"text": prompt}]}],
            "generationConfig": {"temperature": temperature}}
    if as_json:
        body["generationConfig"]["responseMimeType"] = "application/json"
    r = requests.post(url,
                      headers={"x-goog-api-key": API_KEY, "Content-Type": "application/json"},
                      json=body, timeout=180)
    if r.status_code != 200:
        raise RuntimeError("Gemini API error {}: {}".format(r.status_code, r.text[:400]))
    txt = r.json()["candidates"][0]["content"]["parts"][0]["text"]
    if as_json:
        t = re.sub(r"^```(?:json)?", "", txt.strip()).strip()
        t = re.sub(r"```$", "", t).strip()
        return json.loads(t)
    return txt

print("Gemini reachable via", MODEL, "->", gemini("Reply with exactly: OK"))


## 3. Smoke tests
**Two different bars:** test 1 proves Manim renders at all; test 2 proves LaTeX *equations* render. If test 2 fails, your LaTeX install is incomplete — re-run cell 1.

In [ ]:
# Smoke test 1 — does Manim RENDER AT ALL? (no LaTeX involved)
test1 = r'''
from manim import *
class T1(Scene):
    def construct(self):
        self.play(Create(Square()))
        self.wait(0.2)
'''
open("smoke1.py", "w").write(test1)
p = subprocess.run(["manim", "-ql", "smoke1.py", "T1"], capture_output=True, text=True)
if p.returncode == 0 and glob.glob("media/videos/**/T1*.mp4", recursive=True):
    print("PASS: Manim renders.")
else:
    print("FAIL: Manim did not render. Error tail:")
    print(p.stdout[-1200:]); print(p.stderr[-1200:])


In [ ]:
# Smoke test 2 — do LaTeX EQUATIONS render? ("Manim runs" != "equations render")
test2 = r'''
from manim import *
class T2(Scene):
    def construct(self):
        self.play(Write(MathTex(r"\int_0^1 x^2\,dx = \frac{1}{3}")))
        self.wait(0.2)
'''
open("smoke2.py", "w").write(test2)
p = subprocess.run(["manim", "-ql", "smoke2.py", "T2"], capture_output=True, text=True)
if p.returncode == 0 and glob.glob("media/videos/**/T2*.mp4", recursive=True):
    print("PASS: LaTeX equations render in Manim.")
else:
    print("FAIL: equation rendering failed — your LaTeX install is incomplete.")
    print("Re-run cell 1 (the texlive-* packages). Error tail:")
    print(p.stdout[-1200:]); print(p.stderr[-1200:])


## 4. Your inputs
Edit these, then run the rest. `TEMPLATE="auto"` picks a template from your prompt; force it with `"graph"`, `"vector"`, or `"explainer"`.

In [ ]:
# ====== YOUR INPUTS ======
PROMPT      = "Explain the dot product of two vectors and what it means geometrically"
SUBJECT     = "Linear Algebra"   # free text, shown in the site's Video Library
TEMPLATE    = "auto"             # "auto" | "graph" | "vector" | "explainer"
OUTPUT_NAME = ""                 # optional .mp4 name; blank = auto from the prompt


## 5. Vetted scene templates
Three Manim `Scene` skeletons. **Gemini only fills the `CONTENT` data** (what to plot / which vectors / which steps) as validated JSON — it never writes the whole scene. That's what makes it reliable.
- **graph** — a 2D axes + one or more plotted functions + an optional key equation.
- **vector** — vectors on a number plane, with an optional sum (linear-algebra style).
- **explainer** — a sequence of heading + text/equation reveals.

In [ ]:
TEMPLATE_HEADER = r'''
from manim import *
import numpy as np
SAFE = {"np": np, "sin": np.sin, "cos": np.cos, "tan": np.tan, "exp": np.exp,
        "log": np.log, "sqrt": np.sqrt, "pi": np.pi, "abs": abs, "pow": pow}
def _color(name, default):
    return globals().get(str(name).upper(), default)
'''

TEMPLATE_GRAPH = r'''
class GeneratedScene(Scene):
    def construct(self):
        c = CONTENT
        if c.get("title"):
            t = Text(c["title"], font_size=40)
            self.play(Write(t)); self.play(t.animate.to_edge(UP)); self.wait(0.2)
        axes = Axes(x_range=c.get("x_range", [-5, 5, 1]),
                    y_range=c.get("y_range", [-5, 5, 1]),
                    axis_config={"include_numbers": True})
        self.play(Create(axes)); self.wait(0.2)
        for f in c.get("functions", []):
            expr = f.get("expr", "x"); col = _color(f.get("color", "BLUE"), BLUE)
            g = axes.plot(lambda x, e=expr: eval(e, {"__builtins__": {}}, dict(SAFE, x=x)), color=col)
            self.play(Create(g))
            if f.get("label"):
                lab = MathTex(f["label"], color=col).scale(0.8).next_to(axes, UP)
                self.play(Write(lab)); self.wait(0.2)
        if c.get("equation"):
            self.play(Write(MathTex(c["equation"], font_size=42).to_corner(DR)))
        for note in c.get("notes", []):
            n = Text(note, font_size=24).to_edge(DOWN)
            self.play(FadeIn(n, shift=UP)); self.wait(1.2); self.play(FadeOut(n))
        self.wait(1.5)
'''

TEMPLATE_VECTOR = r'''
class GeneratedScene(Scene):
    def construct(self):
        c = CONTENT
        if c.get("title"):
            t = Text(c["title"], font_size=40)
            self.play(Write(t)); self.play(t.animate.to_edge(UP)); self.wait(0.2)
        plane = NumberPlane(x_range=c.get("x_range", [-6, 6, 1]), y_range=c.get("y_range", [-4, 4, 1]))
        self.play(Create(plane)); self.wait(0.2)
        drawn = []
        for v in c.get("vectors", []):
            xy = v.get("coords", [1, 0]); col = _color(v.get("color", "YELLOW"), YELLOW)
            arr = Arrow(plane.c2p(0, 0), plane.c2p(xy[0], xy[1]), buff=0, color=col)
            self.play(GrowArrow(arr))
            if v.get("label"):
                self.play(Write(MathTex(v["label"], color=col).next_to(arr.get_end(), UR, buff=0.1)))
            drawn.append((arr, xy)); self.wait(0.2)
        if c.get("show_sum") and len(drawn) >= 2:
            p1, p2 = drawn[0][1], drawn[1][1]
            s = [p1[0] + p2[0], p1[1] + p2[1]]
            ssum = Arrow(plane.c2p(0, 0), plane.c2p(s[0], s[1]), buff=0, color=GREEN)
            slab = MathTex(c.get("sum_label", "a+b"), color=GREEN).next_to(ssum.get_end(), UR, buff=0.1)
            self.play(GrowArrow(ssum), Write(slab))
        if c.get("equation"):
            self.play(Write(MathTex(c["equation"], font_size=40).to_corner(DR)))
        self.wait(1.5)
'''

TEMPLATE_EXPLAINER = r'''
class GeneratedScene(Scene):
    def construct(self):
        c = CONTENT
        t = Text(c.get("title", ""), font_size=44)
        self.play(Write(t)); self.play(t.animate.to_edge(UP)); self.wait(0.3)
        cur = None
        for sec in c.get("sections", []):
            head = Text(sec.get("heading", ""), font_size=34, color=YELLOW)
            if sec.get("is_math"):
                body = MathTex(sec.get("body", ""), font_size=46)
            else:
                body = Text(sec.get("body", ""), font_size=28, line_spacing=1.1)
                if body.width > 11:
                    body.scale_to_fit_width(11)
            grp = VGroup(head, body).arrange(DOWN, buff=0.5).next_to(t, DOWN, buff=0.8)
            if cur is None:
                self.play(FadeIn(grp, shift=UP))
            else:
                self.play(ReplacementTransform(cur, grp))
            cur = grp; self.wait(1.8)
        self.wait(1.2)
'''

TEMPLATES = {
    "graph": {"body": TEMPLATE_GRAPH, "schema": r'''{
  "title": "short title",
  "x_range": [min, max, step], "y_range": [min, max, step],
  "functions": [ {"expr": "valid Python on x using numpy, e.g. x**2 - 2 or np.sin(x)",
                  "label": "LaTeX label", "color": "BLUE|RED|GREEN|YELLOW|ORANGE|PURPLE"} ],
  "equation": "optional key formula in LaTeX",
  "notes": ["optional short plain-text captions"]
}''', "rules": "Plot functions / graphs / a 2D equation. 'expr' must be valid Python using x and numpy (np, sin, cos, exp, sqrt, pi). Keep each function finite across the whole x_range (avoid division by zero)."},
    "vector": {"body": TEMPLATE_VECTOR, "schema": r'''{
  "title": "short title",
  "x_range": [min, max, step], "y_range": [min, max, step],
  "vectors": [ {"coords": [x, y], "label": "LaTeX e.g. \vec{a}", "color": "YELLOW|RED|BLUE|GREEN"} ],
  "show_sum": true,
  "sum_label": "LaTeX e.g. \vec{a}+\vec{b}",
  "equation": "optional LaTeX"
}''', "rules": "Show vectors on a plane for a linear-algebra idea. Coords must fit inside x_range / y_range. Set show_sum true only when illustrating vector addition."},
    "explainer": {"body": TEMPLATE_EXPLAINER, "schema": r'''{
  "title": "short title",
  "sections": [ {"heading": "short heading", "body": "plain text OR a LaTeX formula", "is_math": false} ]
}''', "rules": "Use 2-4 sections that build the idea in order. Set is_math true only when body is a LaTeX formula (no surrounding text)."},
}
print("Templates ready:", list(TEMPLATES))


## 6. Generate → render → repair
Fills the chosen template with Gemini, renders with Manim, and on failure feeds the **full error + broken content** back to Gemini to fix — up to **3 repair attempts**. If it still fails, it prints the final error and stops (no silent broken file).

In [ ]:
def pick_template(prompt, subject):
    if TEMPLATE != "auto":
        return TEMPLATE
    s = (prompt + " " + subject).lower()
    if any(k in s for k in ["vector", "matrix", "linear algebra", "dot product",
                            "cross product", "span", "basis", "eigen"]):
        return "vector"
    if any(k in s for k in ["graph", "plot", "function", "parabola", "derivative",
                            "integral", "curve", "equation of", "tangent"]):
        return "graph"
    return "explainer"

def build_scene_file(content, tkey):
    return TEMPLATE_HEADER + "\nCONTENT = " + str(content) + "\n" + TEMPLATES[tkey]["body"]

def gen_content(prompt, subject, tkey, broken=None, error=None):
    base = ("You are filling the CONTENT data for a vetted Manim Community scene template. "
            "Return ONLY a JSON object of this shape (no prose, no code fences):\n"
            + TEMPLATES[tkey]["schema"] + "\nRules: " + TEMPLATES[tkey].get("rules", "")
            + "\nTopic to teach: " + prompt + "\nSubject: " + subject
            + "\nBe focused and correct; at most 1-4 functions/vectors/sections; LaTeX must be valid.")
    if broken is not None and error:
        base += ("\n\nThe PREVIOUS CONTENT failed to render. Fix the cause (bad LaTeX, a bad "
                 "'expr', or out-of-range values) and return corrected JSON only.\nPREVIOUS CONTENT:\n"
                 + json.dumps(broken) + "\n\nRENDER ERROR (tail):\n" + error[-1800:])
    return gemini(base, as_json=True, temperature=0.3)

def render(scene_path, quality="-qm"):
    subprocess.run(["rm", "-rf", "media/videos"], capture_output=True, text=True)
    p = subprocess.run(["manim", quality, scene_path, "GeneratedScene"], capture_output=True, text=True)
    log = (p.stdout or "") + "\n" + (p.stderr or "")
    mp4s = glob.glob("media/videos/**/GeneratedScene*.mp4", recursive=True)
    return (p.returncode == 0 and bool(mp4s)), log, (sorted(mp4s)[-1] if mp4s else None)

MAX_REPAIRS = 3
tkey = pick_template(PROMPT, SUBJECT)
print("Template:", tkey)
content = gen_content(PROMPT, SUBJECT, tkey)
final_mp4 = None
for attempt in range(MAX_REPAIRS + 1):           # 1 initial + up to 3 repairs
    stage = "initial render" if attempt == 0 else "repair {}/{}".format(attempt, MAX_REPAIRS)
    print("\n=== " + stage + " ===")
    open("generated_scene.py", "w").write(build_scene_file(content, tkey))
    ok, log, mp4 = render("generated_scene.py")
    if ok:
        final_mp4 = mp4
        print("SUCCESS ->", mp4)
        break
    print("Render failed. Error tail:\n", log[-1200:])
    if attempt == MAX_REPAIRS:
        print("\n!!! Still failing after {} repair attempts. Stopping.".format(MAX_REPAIRS))
        break
    print("Asking Gemini to repair the CONTENT...")
    try:
        content = gen_content(PROMPT, SUBJECT, tkey, broken=content, error=log)
    except Exception as e:
        print("Repair call failed:", e); break

if not final_mp4:
    raise SystemExit("No MP4 produced — see the error above. Refine the prompt/template and re-run.")


## 7. Save the MP4
Saves to **Google Drive** (`MyDrive/ManimVideos/`); falls back to a direct download if Drive isn't mounted.

In [ ]:
import shutil
name = (OUTPUT_NAME.strip() or re.sub(r"[^a-z0-9]+", "-", PROMPT.lower()).strip("-")[:48])
if not name.endswith(".mp4"):
    name += ".mp4"
print("Output name:", name)

saved = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    dst_dir = "/content/drive/MyDrive/ManimVideos"
    os.makedirs(dst_dir, exist_ok=True)
    shutil.copy(final_mp4, os.path.join(dst_dir, name))
    print("Saved to Google Drive:", os.path.join(dst_dir, name))
    saved = True
except Exception as e:
    print("Drive not available ({}). Falling back to download.".format(e))

if not saved:
    try:
        from google.colab import files
        local = "/content/" + name
        shutil.copy(final_mp4, local)
        files.download(local)
    except Exception as e:
        print("Could not auto-download:", e, "\nYour MP4 is here:", final_mp4)

print("\nNext steps:")
print(" 1. Put", name, "into your repo's  videos/  folder.")
print(" 2. Run:   python update_manifest.py")
print(" 3. Commit the .mp4 and videos/manifest.json — it appears in the site's Video Library.")
